<a href="https://colab.research.google.com/github/MichaelG-Coding/CIS3120-Class/blob/main/CIS3120_Gartsbeyn_Michael_LibraryDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Import Libraries and Set The File Paths


In [5]:
import sqlite3
import csv
import urllib.request
import os

#Github raw URLs for the three CSV files
BASE_URL = "https://raw.githubusercontent.com/MichaelG-Coding/CIS3120-Class/main/"

BOOK_URL = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL = BASE_URL + "Loan.csv"

BOOK_PATH = "/content/Book.csv"
MEMBER_PATH = "/content/Member.csv"
LOAN_PATH = "/content/Loan.csv"

DB_PATH = "/content/library.db"

Step 2: Download the CSV Files

In [6]:
for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:
  urllib.request.urlretrieve(url, path)
  print(f"Downloaded: {path}")

Downloaded: /content/Book.csv
Downloaded: /content/Member.csv
Downloaded: /content/Loan.csv


Step 3: Connecting to DataBase and Creating Tables

In [7]:
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON;')

conn.execute("""CREATE TABLE IF NOT EXISTS Book (

    callNo  TEXT    NOT NULL,

    title   TEXT    NOT NULL,

    author  TEXT    NOT NULL,

    PRIMARY KEY (callNo)

);""")

conn.execute("""CREATE TABLE IF NOT EXISTS Member (

    id          INTEGER NOT NULL,

    firstname   TEXT    NOT NULL,

    lastName    TEXT    NOT NULL,

    PRIMARY KEY (id)

);""")

conn.execute("""CREATE TABLE IF NOT EXISTS Loan (

    callNo        TEXT    NOT NULL,

    id            INTEGER NOT NULL,

    dateBorrowed  TEXT    NOT NULL,

    dateReturned  TEXT,

    dateDue       TEXT    NOT NULL,

    PRIMARY KEY (callNo, id, dateBorrowed),

    FOREIGN KEY (callNo) REFERENCES Book(callNo),

    FOREIGN KEY (id)     REFERENCES Member(id)

);""")

conn.commit()

print("Tables created.")
print(conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").fetchall())

Tables created.
[('Book',), ('Loan',), ('Member',)]


Step 4: Load Book.csv into the table

In [8]:
with open(BOOK_PATH, newline='', encoding='utf-8') as f:
  reader = csv.DictReader(f)
  for row in reader:
      conn.execute(
          'INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);',
          (row['callNo'], row['title'], row['author'])
      )
conn.commit()
print('Book rows loaded:', conn.execute('SELECT COUNT(*) FROM Book;').fetchone()[0])

Book rows loaded: 11


Step 5: Load Member.csv into the table

In [9]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            'INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);',
            (int(row['id']), row['firstname'], row['lastName'])
        )
conn.commit()
print('Member rows loaded:', conn.execute('SELECT COUNT(*) FROM Member;').fetchone()[0])

Member rows loaded: 4


Step 6: Load Loan.csv into the table

In [10]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
  reader=csv.DictReader(f)
  for row in reader:
    date_returned = row['dateReturned'] if row['dateReturned'].strip() else None

    conn.execute(
        '''INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
           VALUES (?, ?, ?, ?, ?);''',
          (row['callNo'], int(row['id']),
            row['dateBorrowed'], date_returned, row['dateDue'])
    )
conn.commit()
print('Loan rows loaded:', conn.execute('SELECT COUNT(*) FROM Loan;').fetchone()[0])

Loan rows loaded: 4


Query 1: All the books
Retrieving all the columns from the Book table, ordered alphabetically by authors last name

In [11]:
QUERY1=conn.execute("SELECT*FROM Book ORDER BY author;").fetchall()
print("All books ordered in order:")
for row in QUERY1:
  print(row)

All books ordered in order:
('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


Query 2: Books that have not been returned. Retrive titles of each book as well as names of people who loaned the books.

In [12]:
QUERY2=conn.execute("""SELECT title,firstname,lastname
FROM Loan
JOIN Book ON Loan.callno=Book.callNo
JOIN Member ON Loan.id=Member.id
WHERE dateReturned IS NULL;""").fetchall()
print("Not returned yet:")
print(QUERY2)

Not returned yet:
[("Joe Celko's SQL puzzles & answers", 'David', 'Martin'), ('Medieval medicine and the plague', 'David', 'Martin')]


Query 3: Loan history for a certain book. Find the loan history for book with callNo R 141 E45 2006 to show the name as well as date returned and due and when it was borrowed.

In [13]:
QUERY3=conn.execute("""
SELECT firstname,lastName,dateBorrowed,dateDue,dateReturned
FROM Loan
JOIN Member ON Loan.id=Member.id
WHERE callNo="R 141 E45 2006"
ORDER BY dateBorrowed;
""").fetchall()
print("The loan history shown for 'R 141 E45 2006' is:")
for row in QUERY3:
  print(row)

The loan history shown for 'R 141 E45 2006' is:
('Betty', 'Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David', 'Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


Query 4: Members who have never borrowed a book. Shows the id, firstname, and the lastname for each member that has not borrowed a book.

In [14]:
QUERY4=conn.execute("""
SELECT Member.id,firstname,lastname
FROM Member
LEFT JOIN Loan ON Member.id=Loan.id
WHERE Loan.id IS NULL;
""").fetchall()
print("The members with no borrowing history:")
for row in QUERY4:
  print(row)

The members with no borrowing history:
(4, 'John', 'Martin')


Query 5: Count of Loans per Member showing each members full name and the amount of loans they made (including any completed loans)

In [15]:
QUERY5=conn.execute("""
SELECT firstname,lastname,COUNT(callNo) AS Totalloans
FROM Member
LEFT JOIN Loan ON Member.id=Loan.id
GROUP BY Member.id
ORDER BY Totalloans DESC;
""").fetchall()
print("Count of loans per Member:")
for row in QUERY5:
  print(row)

Count of loans per Member:
('David', 'Martin', 2)
('John', 'Smith', 1)
('Betty', 'Freeman', 1)
('John', 'Martin', 0)


Query 6: The most borrowed books in order. Which Book was borrowed the most?  This is important because it could help determine which book people are most interested in.

In [16]:
QUERY6=conn.execute("""
SELECT title, COUNt(loan.callNo) as timesTaken
FROM Book
JOIN LOAN ON book.callNo=loan.callNo
GROUP BY title
ORDER BY timesTaken DESC;
""").fetchall()
print("The most borrowed book in order is:")
for row in QUERY6:
  print(row)

The most borrowed book in order is:
('Medieval medicine and the plague', 2)
('Medieval women', 1)
("Joe Celko's SQL puzzles & answers", 1)


Markdown Summary

One data quality observation I made was that the values in dateReturned were blank and while in our eyes this means that it was never returned, python would just see it as a blank string instead of NULL causing us to have to write code to change them to NULL. One limitation of this dataset for a real library system is that there are very few variables shown in the dataset which means theres little use for it in a library. It could use a variable such as the genre of a book to add more details to the dataset for the library.